# Video Game Market Segmentation: Business Analysis and Strategic Insights

## Comprehensive Documentation and Cluster Interpretation

---

### Executive Summary

This notebook provides comprehensive business analysis and strategic interpretation of the discovered video game market segments. The analysis transforms technical clustering results into actionable business intelligence for game developers, publishers, and industry stakeholders.

### Document Structure

1. **Data Loading and Validation** - Import clustering results and verify data quality
2. **Cluster Characterization** - Detailed profiling of each market segment
3. **Success Factor Analysis** - Identification of key drivers within each segment
4. **Competitive Landscape** - Market positioning and competition assessment
5. **Strategic Recommendations** - Actionable guidance for market entry and positioning
6. **Visualization Dashboard** - Comprehensive visual analytics
7. **Final Documentation** - Executive report generation

---

## 1. Data Loading and Validation

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

# Configure environment
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', 100)
pd.set_option('display.precision', 3)

print("Environment configured successfully")

In [ ]:
# Load clustering results
print("Loading clustered data and results...\n")

# Main datasets
df = pd.read_csv('df_with_clusters_enhanced.csv')
profile_df = pd.read_csv('cluster_profiles_enhanced.csv')

# Load metadata
metadata = {}
with open('model_metadata_enhanced.txt', 'r') as f:
    for line in f:
        if ':' in line:
            key, value = line.strip().split(': ', 1)
            try:
                metadata[key] = float(value) if '.' in value else int(value) if value.isdigit() else value
            except:
                metadata[key] = value

# Extract key parameters
n_clusters = int(metadata.get('optimal_k', len(profile_df)))
silhouette_score = float(metadata.get('silhouette_score', 0))
algorithm = metadata.get('algorithm', 'K-Means')
n_samples = len(df)

# Data validation
print("Data Validation Summary:")
print(f"  Total games clustered: {n_samples:,}")
print(f"  Number of clusters: {n_clusters}")
print(f"  Clustering algorithm: {algorithm}")
print(f"  Silhouette score: {silhouette_score:.4f}")
print(f"  Quality assessment: {'Excellent' if silhouette_score >= 0.55 else 'Good' if silhouette_score >= 0.50 else 'Acceptable'}")
print(f"\nCluster size distribution:")
cluster_sizes = df['cluster'].value_counts().sort_index()
for cluster_id, size in cluster_sizes.items():
    pct = (size / n_samples) * 100
    print(f"  Cluster {cluster_id}: {size:,} games ({pct:.1f}%)")

---

## 2. Cluster Characterization and Naming

### 2.1 Intelligent Cluster Naming Framework

Clusters are named based on three key dimensions:
1. **Market Scope:** Global vs Regional focus (measured by HHI index)
2. **Performance Tier:** Premium, Mid-Tier, or Indie/Small (based on sales quartiles)
3. **Geographic Focus:** Dominant region (NA, JP, EU, or Balanced)

This multi-dimensional framework ensures meaningful, descriptive cluster names that reflect actual market positioning.

In [ ]:
# Generate intelligent cluster names and descriptions
print("Generating cluster characterizations...\n")

cluster_names = {}
cluster_descriptions = {}
cluster_interpretations = {}

for idx, row in profile_df.iterrows():
    c = int(row['Cluster'])
    
    # Extract key metrics
    avg_sales = row['Avg_Sales']
    med_sales = row['Med_Sales']
    hhi = row['HHI']
    na_ratio = row['NA_Ratio']
    jp_ratio = row['JP_Ratio']
    pal_ratio = row['PAL_Ratio']
    hit_rate = row['Hit_Rate']
    global_pct = row.get('Global_Game_%', 0)
    
    # Determine market scope
    if hhi < 0.45:
        market_scope = "Global"
        scope_detail = "balanced distribution across all major markets"
    elif hhi > 0.75:
        market_scope = "Highly Regional"
        scope_detail = "concentrated in a single dominant market"
    else:
        market_scope = "Regional"
        scope_detail = "focused on 1-2 primary markets"
    
    # Determine geographic focus
    max_region_ratio = max(na_ratio, jp_ratio, pal_ratio)
    if na_ratio == max_region_ratio and na_ratio > 0.5:
        geo_focus = "North American"
        primary_market = "NA"
    elif jp_ratio == max_region_ratio and jp_ratio > 0.5:
        geo_focus = "Japanese"
        primary_market = "JP"
    elif pal_ratio == max_region_ratio and pal_ratio > 0.4:
        geo_focus = "European"
        primary_market = "EU"
    else:
        geo_focus = "Multi-Regional"
        primary_market = "Global"
    
    # Determine performance tier
    sales_quartiles = profile_df['Avg_Sales'].quantile([0.33, 0.66])
    if avg_sales > sales_quartiles[0.66]:
        tier = "Premium"
        tier_detail = "high-budget, high-revenue titles"
    elif avg_sales > sales_quartiles[0.33]:
        tier = "Mid-Market"
        tier_detail = "moderate-budget commercial titles"
    else:
        tier = "Indie/Small-Scale"
        tier_detail = "lower-budget niche titles"
    
    # Generate cluster name
    if tier == "Premium":
        if market_scope == "Global":
            name = "Global Premium Market"
            icon = "🌍"
        else:
            name = f"{geo_focus} Premium Market"
            icon = "🏆"
    elif tier == "Mid-Market":
        if market_scope == "Global":
            name = "Global Mid-Tier Market"
            icon = "🎮"
        else:
            name = f"{geo_focus} Mid-Tier Market"
            icon = "📊"
    else:
        if market_scope == "Highly Regional":
            name = f"{geo_focus} Niche/Specialist Market"
            icon = "🎯"
        else:
            name = "Emerging/Indie Market"
            icon = "💎"
    
    # Generate description
    description = f"{tier_detail.capitalize()} with {scope_detail}. "
    description += f"Primary market: {primary_market}. "
    description += f"Average sales: ${avg_sales:.2f}M. "
    description += f"Hit rate: {hit_rate:.1f}%."
    
    # Generate detailed interpretation
    interpretation = f"""This segment represents {tier_detail} characterized by {scope_detail}. 
Games in this cluster typically target the {geo_focus.lower()} market{'s' if 'Multi' not in geo_focus else ''}, 
with an average sales performance of ${avg_sales:.2f}M (median: ${med_sales:.2f}M). 
Approximately {hit_rate:.1f}% of games in this segment achieve 'hit' status (top 25% sales). 
Market concentration index (HHI) of {hhi:.2f} indicates a {market_scope.lower()} market approach."""
    
    cluster_names[c] = f"{icon} {name}"
    cluster_descriptions[c] = description
    cluster_interpretations[c] = interpretation.replace('\n', ' ')

print("Cluster characterization complete.")

### 2.2 Detailed Cluster Profiles

In [ ]:
# Display comprehensive cluster profiles
print("=" * 100)
print("DISCOVERED MARKET SEGMENTS: DETAILED PROFILES")
print("=" * 100)
print()

for c in sorted(cluster_names.keys()):
    row = profile_df[profile_df['Cluster'] == c].iloc[0]
    
    print(f"Cluster {c}: {cluster_names[c]}")
    print("=" * 100)
    print()
    print(f"Description:")
    print(f"  {cluster_descriptions[c]}")
    print()
    print(f"Interpretation:")
    print(f"  {cluster_interpretations[c]}")
    print()
    print(f"Key Metrics:")
    print(f"  Market Size:           {int(row['Size']):,} games ({row['Size_%']:.1f}% of total market)")
    print(f"  Sales Performance:     ${row['Avg_Sales']:.2f}M average, ${row['Med_Sales']:.2f}M median")
    print(f"  Quality Level:         {row['Avg_Critic']:.1f}/100 critic score")
    print(f"  Average Game Age:      {row['Avg_Game_Age']:.1f} years")
    print(f"  Hit Success Rate:      {row['Hit_Rate']:.1f}% achieve top 25% sales")
    print()
    print(f"Regional Distribution:")
    print(f"  North America:         {row['NA_Ratio']*100:.1f}%")
    print(f"  Japan:                 {row['JP_Ratio']*100:.1f}%")
    print(f"  Europe (PAL):          {row['PAL_Ratio']*100:.1f}%")
    print(f"  Market Concentration:  {row['HHI']:.3f} ({'Global' if row['HHI'] < 0.5 else 'Regional'})")
    print()
    print(f"Popular Characteristics:")
    print(f"  Most Common Genre:     {row['Top_Genre']}")
    print(f"  Most Common Platform:  {row['Top_Platform']}")
    if 'Global_Game_%' in row:
        print(f"  Global Games:          {row['Global_Game_%']:.1f}%")
    print()
    print("-" * 100)
    print()

---

## 3. Success Factor Analysis

### 3.1 Methodology

For each cluster, we identify key success factors by comparing the top 25% performers (successful games) against the bottom 75% (unsuccessful games) across multiple dimensions:

- **Quality Impact:** Difference in critic scores
- **Temporal Factor:** Age difference (newer vs. older games)
- **Regional Strategy:** Market concentration differences
- **Publisher Influence:** Reputation and track record impact
- **Genre/Platform Optimization:** Most successful combinations

In [ ]:
# Perform success factor analysis for each cluster
print("=" * 100)
print("SUCCESS FACTOR ANALYSIS BY SEGMENT")
print("=" * 100)
print()

success_insights = {}

for c in sorted(cluster_names.keys()):
    cluster_data = df[df['cluster'] == c].copy()
    
    # Define success threshold (top 25% within cluster)
    threshold = cluster_data['total_sales'].quantile(0.75)
    
    successful = cluster_data[cluster_data['total_sales'] >= threshold]
    unsuccessful = cluster_data[cluster_data['total_sales'] < threshold]
    
    print(f"Cluster {c}: {cluster_names[c]}")
    print("=" * 100)
    print()
    
    if len(successful) >= 10 and len(unsuccessful) >= 10:
        print(f"Success Threshold: ${threshold:.2f}M (top 25% of cluster)")
        print(f"Successful games: {len(successful)} | Unsuccessful games: {len(unsuccessful)}")
        print()
        
        # 1. Quality Impact Analysis
        critic_diff = successful['critic_score'].mean() - unsuccessful['critic_score'].mean()
        t_stat, p_value = stats.ttest_ind(successful['critic_score'].dropna(), 
                                          unsuccessful['critic_score'].dropna())
        
        print(f"Quality Impact Analysis:")
        print(f"  Successful games average: {successful['critic_score'].mean():.1f}/100")
        print(f"  Unsuccessful games average: {unsuccessful['critic_score'].mean():.1f}/100")
        print(f"  Difference: {critic_diff:+.2f} points")
        print(f"  Statistical significance: {'Yes (p<0.05)' if p_value < 0.05 else 'No (p≥0.05)'}")
        if abs(critic_diff) > 1.0:
            print(f"  Interpretation: Quality is a CRITICAL success factor in this segment")
        elif abs(critic_diff) > 0.5:
            print(f"  Interpretation: Quality has MODERATE impact on success")
        else:
            print(f"  Interpretation: Quality has LIMITED impact (other factors more important)")
        print()
        
        # 2. Temporal Analysis
        age_diff = unsuccessful['game_age'].mean() - successful['game_age'].mean()
        print(f"Temporal Analysis:")
        print(f"  Successful games average age: {successful['game_age'].mean():.1f} years")
        print(f"  Unsuccessful games average age: {unsuccessful['game_age'].mean():.1f} years")
        print(f"  Age difference: {age_diff:+.1f} years")
        if age_diff > 1.0:
            print(f"  Interpretation: Newer games have SIGNIFICANT advantage in this segment")
        elif age_diff < -1.0:
            print(f"  Interpretation: Older games maintain value (evergreen segment)")
        else:
            print(f"  Interpretation: Age has minimal impact on success")
        print()
        
        # 3. Regional Strategy Analysis
        hhi_diff = successful['sales_concentration_hhi'].mean() - unsuccessful['sales_concentration_hhi'].mean()
        print(f"Regional Strategy Analysis:")
        print(f"  Successful games HHI: {successful['sales_concentration_hhi'].mean():.3f}")
        print(f"  Unsuccessful games HHI: {unsuccessful['sales_concentration_hhi'].mean():.3f}")
        print(f"  Difference: {hhi_diff:+.3f}")
        if hhi_diff > 0.05:
            print(f"  Interpretation: Focused regional strategy works better (target 1-2 markets)")
        elif hhi_diff < -0.05:
            print(f"  Interpretation: Balanced global distribution works better (target all markets)")
        else:
            print(f"  Interpretation: Regional strategy has minimal impact")
        print()
        
        # 4. Publisher Influence
        if 'pub_avg_sales' in successful.columns:
            pub_impact = successful['pub_avg_sales'].mean() - unsuccessful['pub_avg_sales'].mean()
            print(f"Publisher Reputation Analysis:")
            print(f"  Successful games - publisher avg sales: ${successful['pub_avg_sales'].mean():.2f}M")
            print(f"  Unsuccessful games - publisher avg sales: ${unsuccessful['pub_avg_sales'].mean():.2f}M")
            print(f"  Difference: ${pub_impact:+.2f}M")
            if pub_impact > 0.3:
                print(f"  Interpretation: Established publishers have SIGNIFICANT advantage")
            elif pub_impact > 0.1:
                print(f"  Interpretation: Publisher reputation provides MODERATE advantage")
            else:
                print(f"  Interpretation: Publisher reputation has LIMITED impact (open to new publishers)")
            print()
        
        # 5. Genre & Platform Success Factors
        top_genres = successful['genre_original'].value_counts().head(5)
        print(f"Most Successful Genres:")
        for i, (genre, count) in enumerate(top_genres.items(), 1):
            pct = (count / len(successful)) * 100
            print(f"  {i}. {genre}: {count} games ({pct:.1f}% of successful)")
        print()
        
        top_platforms = successful['console_original'].value_counts().head(5)
        print(f"Most Successful Platforms:")
        for i, (platform, count) in enumerate(top_platforms.items(), 1):
            pct = (count / len(successful)) * 100
            print(f"  {i}. {platform}: {count} games ({pct:.1f}% of successful)")
        print()
        
        # Store insights for recommendations
        success_insights[c] = {
            'threshold': threshold,
            'critic_diff': critic_diff,
            'age_diff': age_diff,
            'hhi_diff': hhi_diff,
            'pub_impact': pub_impact if 'pub_avg_sales' in successful.columns else 0,
            'top_genres': top_genres.index.tolist()[:3],
            'top_platforms': top_platforms.index.tolist()[:3]
        }
    else:
        print(f"Insufficient data for detailed success analysis (cluster size: {len(cluster_data)})")
        print()
    
    print("-" * 100)
    print()

---

## 4. Competitive Landscape Analysis

Understanding the competitive dynamics within each segment is crucial for market entry and positioning strategy.

In [ ]:
# Analyze competitive landscape for each cluster
print("=" * 100)
print("COMPETITIVE LANDSCAPE ANALYSIS")
print("=" * 100)
print()

for c in sorted(cluster_names.keys()):
    cluster_data = df[df['cluster'] == c]
    row = profile_df[profile_df['Cluster'] == c].iloc[0]
    
    print(f"Cluster {c}: {cluster_names[c]}")
    print("=" * 100)
    print()
    
    # Market Size and Share
    market_share = row['Size_%']
    print(f"Market Size:")
    print(f"  Total games: {int(row['Size']):,}")
    print(f"  Market share: {market_share:.1f}% of total video game market")
    
    if market_share > 30:
        competitive_intensity = "VERY HIGH"
        entry_difficulty = "Extremely challenging - requires significant differentiation"
    elif market_share > 20:
        competitive_intensity = "HIGH"
        entry_difficulty = "Challenging - strong value proposition needed"
    elif market_share > 10:
        competitive_intensity = "MODERATE"
        entry_difficulty = "Moderate - solid execution required"
    else:
        competitive_intensity = "LOW"
        entry_difficulty = "Favorable - opportunity for new entrants"
    
    print(f"  Competitive intensity: {competitive_intensity}")
    print(f"  Market entry assessment: {entry_difficulty}")
    print()
    
    # Recent Market Activity (last 3 years)
    recent_games = cluster_data[cluster_data['game_age'] <= 3]
    if len(recent_games) > 0:
        recent_pct = (len(recent_games) / len(cluster_data)) * 100
        print(f"Recent Market Activity (last 3 years):")
        print(f"  Recent releases: {len(recent_games)} games ({recent_pct:.1f}% of segment)")
        
        if recent_pct > 30:
            market_dynamics = "GROWING - Active market with frequent new releases"
        elif recent_pct > 15:
            market_dynamics = "STABLE - Consistent release activity"
        else:
            market_dynamics = "DECLINING - Limited recent activity"
        
        print(f"  Market dynamics: {market_dynamics}")
        print()
        
        # Publisher Concentration in Recent Releases
        if len(recent_games) >= 10:
            top_publishers = recent_games['publisher_original'].value_counts().head(5)
            top3_concentration = (top_publishers.head(3).sum() / len(recent_games)) * 100
            
            print(f"Publisher Concentration (recent releases):")
            print(f"  Top 3 publishers control: {top3_concentration:.1f}% of recent releases")
            
            if top3_concentration > 50:
                market_structure = "HIGHLY CONCENTRATED - Dominated by few major publishers"
            elif top3_concentration > 30:
                market_structure = "MODERATELY CONCENTRATED - Mix of major and mid-tier publishers"
            else:
                market_structure = "FRAGMENTED - Diverse publisher landscape"
            
            print(f"  Market structure: {market_structure}")
            print()
            print(f"  Leading publishers:")
            for i, (pub, count) in enumerate(top_publishers.items(), 1):
                pct = (count / len(recent_games)) * 100
                print(f"    {i}. {pub}: {count} games ({pct:.1f}%)")
            print()
    
    # Success Difficulty Assessment
    hit_rate = row['Hit_Rate']
    print(f"Success Probability:")
    print(f"  Hit rate (top 25% sales): {hit_rate:.1f}%")
    print(f"  Average sales: ${row['Avg_Sales']:.2f}M")
    print(f"  Median sales: ${row['Med_Sales']:.2f}M")
    
    if hit_rate > 20 and row['Avg_Sales'] > 0.5:
        opportunity = "HIGH POTENTIAL - Good hit rate with strong sales performance"
    elif hit_rate > 15:
        opportunity = "MODERATE POTENTIAL - Reasonable success probability"
    else:
        opportunity = "CHALLENGING - Lower success probability, requires careful positioning"
    
    print(f"  Market opportunity: {opportunity}")
    print()
    
    print("-" * 100)
    print()

---

## 5. Strategic Recommendations

### 5.1 Segment-Specific Go-to-Market Strategies

Based on cluster characteristics, success factors, and competitive analysis, we provide actionable recommendations for each market segment.

In [ ]:
# Generate strategic recommendations for each cluster
print("=" * 100)
print("STRATEGIC RECOMMENDATIONS BY MARKET SEGMENT")
print("=" * 100)
print()

for c in sorted(cluster_names.keys()):
    row = profile_df[profile_df['Cluster'] == c].iloc[0]
    insights = success_insights.get(c, {})
    
    print(f"Cluster {c}: {cluster_names[c]}")
    print("=" * 100)
    print()
    
    # 1. Budget Recommendation
    avg_sales = row['Avg_Sales']
    if avg_sales > 2.0:
        budget_range = "$15M - $100M+"
        budget_tier = "AAA / Premium"
        budget_note = "Requires major publisher backing and extensive resources"
    elif avg_sales > 0.8:
        budget_range = "$3M - $15M"
        budget_tier = "AA / Mid-Budget"
        budget_note = "Achievable with mid-tier publisher or well-funded independent studio"
    elif avg_sales > 0.3:
        budget_range = "$500K - $3M"
        budget_tier = "Low-Budget / Small Publisher"
        budget_note = "Appropriate for indie studios with external funding"
    else:
        budget_range = "$50K - $500K"
        budget_tier = "Micro / Indie"
        budget_note = "Bootstrap or crowdfunding viable options"
    
    print(f"Development Budget:")
    print(f"  Recommended range: {budget_range}")
    print(f"  Budget tier: {budget_tier}")
    print(f"  Note: {budget_note}")
    print()
    
    # 2. Quality Requirements
    avg_critic = row['Avg_Critic']
    critic_diff = insights.get('critic_diff', 0)
    
    if avg_critic > 80 or critic_diff > 1.0:
        quality_req = "CRITICAL - High quality is essential for success"
        quality_target = f"{avg_critic + 5:.0f}+/100"
    elif avg_critic > 70 or critic_diff > 0.5:
        quality_req = "IMPORTANT - Solid quality necessary but not sufficient"
        quality_target = f"{avg_critic:.0f}+/100"
    else:
        quality_req = "MODERATE - Other factors more important than raw quality"
        quality_target = f"{avg_critic - 5:.0f}+/100 (minimum acceptable)"
    
    print(f"Quality Requirements:")
    print(f"  Segment average: {avg_critic:.1f}/100")
    print(f"  Recommended target: {quality_target}")
    print(f"  Quality impact: {quality_req}")
    print()
    
    # 3. Genre and Platform Strategy
    print(f"Genre Strategy:")
    print(f"  Primary recommendation: {row['Top_Genre']}")
    if 'top_genres' in insights:
        alt_genres = [g for g in insights['top_genres'] if g != row['Top_Genre']]
        if alt_genres:
            print(f"  Alternative genres: {', '.join(alt_genres)}")
    print()
    
    print(f"Platform Strategy:")
    print(f"  Primary recommendation: {row['Top_Platform']}")
    if 'top_platforms' in insights:
        alt_platforms = [p for p in insights['top_platforms'] if p != row['Top_Platform']]
        if alt_platforms:
            print(f"  Alternative platforms: {', '.join(alt_platforms)}")
    print()
    
    # 4. Regional Strategy
    na_ratio, jp_ratio, pal_ratio = row['NA_Ratio'], row['JP_Ratio'], row['PAL_Ratio']
    hhi = row['HHI']
    hhi_diff = insights.get('hhi_diff', 0)
    
    print(f"Regional Strategy:")
    print(f"  Current distribution: NA {na_ratio*100:.0f}% | JP {jp_ratio*100:.0f}% | EU {pal_ratio*100:.0f}%")
    
    if hhi_diff > 0.05:
        strategy_type = "FOCUSED"
        strategy_desc = "Concentrate on 1-2 primary markets for maximum impact"
    elif hhi_diff < -0.05:
        strategy_type = "BALANCED"
        strategy_desc = "Pursue balanced multi-regional approach"
    else:
        strategy_type = "FLEXIBLE"
        strategy_desc = "Adapt based on genre and platform strengths"
    
    print(f"  Recommended approach: {strategy_type}")
    print(f"  Rationale: {strategy_desc}")
    
    # Specific regional recommendations
    if na_ratio > 0.5:
        print(f"  Primary target: North America & Europe")
        print(f"    - Localization: Full English, consider French/Spanish")
        print(f"    - Marketing: Focus on Xbox, PlayStation, PC platforms")
        print(f"    - Themes: Western-friendly narratives and mechanics")
    elif jp_ratio > 0.5:
        print(f"  Primary target: Japan & East Asia")
        print(f"    - Localization: Japanese essential, consider Korean/Chinese")
        print(f"    - Marketing: Focus on PlayStation, Nintendo platforms")
        print(f"    - Themes: Asian market preferences and cultural elements")
    elif hhi < 0.5:
        print(f"  Primary target: Global (all major markets)")
        print(f"    - Localization: Multi-language from day one")
        print(f"    - Marketing: Coordinated global campaign")
        print(f"    - Themes: Universally appealing content")
    print()
    
    # 5. Competitive Positioning
    market_share = row['Size_%']
    hit_rate = row['Hit_Rate']
    
    print(f"Competitive Positioning:")
    print(f"  Market segment size: {market_share:.1f}% of total market")
    print(f"  Hit success rate: {hit_rate:.1f}%")
    
    if market_share > 30:
        print(f"  Competition level: VERY HIGH")
        print(f"  Entry strategy: Strong differentiation required - unique IP, innovative mechanics, or established franchise")
    elif market_share > 20:
        print(f"  Competition level: HIGH")
        print(f"  Entry strategy: Clear value proposition needed - genre innovation or superior execution")
    elif market_share > 10:
        print(f"  Competition level: MODERATE")
        print(f"  Entry strategy: Solid quality and targeted marketing sufficient")
    else:
        print(f"  Competition level: LOW")
        print(f"  Entry strategy: Favorable conditions - opportunity for market share capture")
    print()
    
    # 6. Publisher Recommendations
    pub_impact = insights.get('pub_impact', 0)
    
    print(f"Publisher Strategy:")
    if pub_impact > 0.3:
        print(f"  Recommendation: Pursue established publisher partnership")
        print(f"  Rationale: Publisher reputation has significant impact on success ({pub_impact:+.2f}M avg sales difference)")
        print(f"  Target publishers: Major publishers with track record in this segment")
    elif pub_impact > 0.1:
        print(f"  Recommendation: Mid-tier publisher or well-funded self-publishing")
        print(f"  Rationale: Publisher reputation provides moderate advantage")
        print(f"  Target publishers: Specialized publishers or self-publish with marketing budget")
    else:
        print(f"  Recommendation: Self-publishing viable")
        print(f"  Rationale: Publisher reputation has limited impact - execution matters more")
        print(f"  Approach: Independent release with digital distribution")
    print()
    
    # 7. Financial Projections
    print(f"Financial Expectations:")
    print(f"  Average sales: ${row['Avg_Sales']:.2f}M")
    print(f"  Median sales: ${row['Med_Sales']:.2f}M (50th percentile)")
    threshold = insights.get('threshold', row['Avg_Sales'] * 1.5)
    print(f"  Success threshold: ${threshold:.2f}M (75th percentile)")
    print(f"  Hit probability: {hit_rate:.1f}% chance of top 25% performance")
    print()
    
    print("-" * 100)
    print()

---

## 6. Comprehensive Visualization Dashboard

Visual analytics to support decision-making and presentation.

In [ ]:
# Create comprehensive business analytics dashboard
print("Generating comprehensive visualization dashboard...")

fig = plt.figure(figsize=(22, 16))
gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.35)

colors = plt.cm.Set3(np.arange(n_clusters))

# 1. Market Share Distribution
ax = fig.add_subplot(gs[0, 0])
cluster_labels = [cluster_names[i].replace('🌍 ', '').replace('🏆 ', '').replace('🎮 ', '').replace('📊 ', '').replace('🎯 ', '').replace('💎 ', '') 
                  for i in range(n_clusters)]
sizes = profile_df['Size']
ax.pie(sizes, labels=cluster_labels, autopct='%1.1f%%', colors=colors, 
       startangle=90, textprops={'fontsize': 8, 'fontweight': 'bold'})
ax.set_title('Market Share Distribution by Segment', fontsize=12, fontweight='bold', pad=10)

# 2. Sales Performance Comparison
ax = fig.add_subplot(gs[0, 1:])
x_pos = np.arange(n_clusters)
width = 0.35
bars1 = ax.bar(x_pos - width/2, profile_df['Avg_Sales'], width, 
               label='Average', color=colors, edgecolor='black', linewidth=1.5)
bars2 = ax.bar(x_pos + width/2, profile_df['Med_Sales'], width,
               label='Median', color=colors, alpha=0.6, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Cluster', fontsize=11, fontweight='bold')
ax.set_ylabel('Sales ($M)', fontsize=11, fontweight='bold')
ax.set_title('Sales Distribution by Segment (Average vs Median)', 
             fontsize=12, fontweight='bold', pad=10)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'C{i}' for i in range(n_clusters)])
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# 3. Regional Distribution Heatmap
ax = fig.add_subplot(gs[1, :2])
regional_data = profile_df[['NA_Ratio', 'JP_Ratio', 'PAL_Ratio']].values * 100
sns.heatmap(regional_data, annot=True, fmt='.1f', cmap='YlOrRd',
           xticklabels=['North America', 'Japan', 'Europe'],
           yticklabels=cluster_labels,
           ax=ax, linewidths=2, cbar_kws={'label': 'Sales %'}, 
           vmin=0, vmax=100, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax.set_title('Regional Sales Distribution by Segment', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Region', fontsize=11, fontweight='bold')
ax.set_ylabel('Market Segment', fontsize=11, fontweight='bold')

# 4. Quality vs Sales Performance
ax = fig.add_subplot(gs[1, 2])
bubble_sizes = profile_df['Size'] * 2
scatter = ax.scatter(profile_df['Avg_Critic'], profile_df['Avg_Sales'],
                    s=bubble_sizes, c=colors, alpha=0.7, 
                    edgecolors='black', linewidth=2)
for i, row in profile_df.iterrows():
    ax.annotate(f"C{int(row['Cluster'])}", 
                (row['Avg_Critic'], row['Avg_Sales']),
                fontsize=10, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle', facecolor='white', alpha=0.7))
ax.set_xlabel('Average Critic Score', fontsize=11, fontweight='bold')
ax.set_ylabel('Average Sales ($M)', fontsize=11, fontweight='bold')
ax.set_title('Quality vs Commercial Performance\n(bubble size = market size)', 
             fontsize=12, fontweight='bold', pad=10)
ax.grid(alpha=0.3)

# 5. Hit Rate by Segment
ax = fig.add_subplot(gs[2, 0])
bars = ax.barh(range(n_clusters), profile_df['Hit_Rate'], 
               color=colors, edgecolor='black', linewidth=2)
ax.set_yticks(range(n_clusters))
ax.set_yticklabels([f'C{i}' for i in range(n_clusters)])
ax.set_xlabel('Hit Rate (%)', fontsize=11, fontweight='bold')
ax.set_ylabel('Cluster', fontsize=11, fontweight='bold')
ax.set_title('Success Rate (Top 25% Sales)\nby Segment', 
             fontsize=12, fontweight='bold', pad=10)
ax.grid(axis='x', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, profile_df['Hit_Rate'])):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, 
            f"{val:.1f}%", va='center', fontweight='bold')

# 6. Market Concentration (HHI)
ax = fig.add_subplot(gs[2, 1])
bars = ax.bar(range(n_clusters), profile_df['HHI'], 
              color=colors, edgecolor='black', linewidth=2)
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, 
           alpha=0.7, label='Global/Regional Threshold')
ax.set_xticks(range(n_clusters))
ax.set_xticklabels([f'C{i}' for i in range(n_clusters)])
ax.set_xlabel('Cluster', fontsize=11, fontweight='bold')
ax.set_ylabel('HHI Index', fontsize=11, fontweight='bold')
ax.set_title('Market Concentration\n(HHI < 0.5 = Global)', 
             fontsize=12, fontweight='bold', pad=10)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1])

# 7. Average Game Age
ax = fig.add_subplot(gs[2, 2])
bars = ax.bar(range(n_clusters), profile_df['Avg_Game_Age'],
              color=colors, edgecolor='black', linewidth=2)
ax.set_xticks(range(n_clusters))
ax.set_xticklabels([f'C{i}' for i in range(n_clusters)])
ax.set_xlabel('Cluster', fontsize=11, fontweight='bold')
ax.set_ylabel('Average Age (years)', fontsize=11, fontweight='bold')
ax.set_title('Average Game Age\nby Segment', 
             fontsize=12, fontweight='bold', pad=10)
ax.grid(axis='y', alpha=0.3)

# 8. Top Genres Distribution
ax = fig.add_subplot(gs[3, :])
genre_data = df.groupby(['cluster', 'genre_original']).size().unstack(fill_value=0)
genre_data = genre_data[genre_data.sum(axis=0).nlargest(10).index]
genre_data_pct = genre_data.div(genre_data.sum(axis=1), axis=0) * 100

x = np.arange(len(genre_data.columns))
width = 0.15
for i in range(min(n_clusters, 5)):  # Limit to 5 clusters for readability
    if i in genre_data_pct.index:
        offset = (i - (n_clusters-1)/2) * width
        ax.bar(x + offset, genre_data_pct.loc[i], width, 
               label=f'C{i}', color=colors[i], edgecolor='black', linewidth=1)

ax.set_xlabel('Genre', fontsize=11, fontweight='bold')
ax.set_ylabel('Percentage within Cluster (%)', fontsize=11, fontweight='bold')
ax.set_title('Top Genre Distribution by Segment', fontsize=12, fontweight='bold', pad=10)
ax.set_xticks(x)
ax.set_xticklabels(genre_data.columns, rotation=45, ha='right')
ax.legend(title='Cluster', loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Video Game Market Segmentation Analysis\n{n_clusters} Clusters | Silhouette Score: {silhouette_score:.4f} | Algorithm: {algorithm}',
            fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
plt.savefig('business_analytics_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("Dashboard saved: business_analytics_dashboard.png")

---

## 7. Executive Documentation Export

Generate comprehensive final report for stakeholders.

In [ ]:
# Generate comprehensive business documentation
print("Generating executive documentation...\n")

doc_content = f"""
{'='*100}
VIDEO GAME MARKET SEGMENTATION - EXECUTIVE REPORT
{'='*100}

EXECUTIVE SUMMARY
{'-'*100}

This analysis employed advanced clustering techniques to segment the video game market
into {n_clusters} distinct, actionable market segments. The analysis processed {n_samples:,}
commercial video games using {algorithm} clustering algorithm, achieving a Silhouette
score of {silhouette_score:.4f}, indicating {'excellent' if silhouette_score >= 0.55 else 'good' if silhouette_score >= 0.50 else 'acceptable'} cluster quality.

KEY FINDINGS
{'-'*100}

1. Market Structure: {n_clusters} distinct segments with varying characteristics
2. Geographic Patterns: Clear regional preferences across segments
3. Quality Requirements: Variable importance of quality scores by segment
4. Publisher Influence: Significant in some segments, minimal in others
5. Success Factors: Segment-specific drivers identified

DISCOVERED MARKET SEGMENTS
{'-'*100}
"""

for c in sorted(cluster_names.keys()):
    row = profile_df[profile_df['Cluster'] == c].iloc[0]
    insights = success_insights.get(c, {})
    
    doc_content += f"""
SEGMENT {c+1}: {cluster_names[c].replace(cluster_names[c].split()[0], '').strip()}
{'-'*100}

Description:
{cluster_descriptions[c]}

Interpretation:
{cluster_interpretations[c]}

Market Metrics:
  • Market Share: {row['Size_%']:.1f}% ({int(row['Size']):,} games)
  • Average Sales: ${row['Avg_Sales']:.2f}M
  • Median Sales: ${row['Med_Sales']:.2f}M
  • Quality Level: {row['Avg_Critic']:.1f}/100
  • Hit Rate: {row['Hit_Rate']:.1f}%

Regional Distribution:
  • North America: {row['NA_Ratio']*100:.1f}%
  • Japan: {row['JP_Ratio']*100:.1f}%
  • Europe: {row['PAL_Ratio']*100:.1f}%
  • Market Type: {'Global' if row['HHI'] < 0.5 else 'Regional'} (HHI: {row['HHI']:.3f})

Popular Attributes:
  • Top Genre: {row['Top_Genre']}
  • Top Platform: {row['Top_Platform']}
  • Average Age: {row['Avg_Game_Age']:.1f} years

Success Factors:"""
    
    if insights:
        doc_content += f"""
  • Success Threshold: ${insights['threshold']:.2f}M (top 25%)
  • Quality Impact: {insights['critic_diff']:+.2f} points {'(critical)' if abs(insights['critic_diff']) > 1.0 else '(moderate)' if abs(insights['critic_diff']) > 0.5 else '(limited)'}
  • Age Factor: {insights['age_diff']:+.1f} years {'(newer games favored)' if insights['age_diff'] > 1 else '(age neutral)' if abs(insights['age_diff']) <= 1 else '(older games retain value)'}
  • Regional Strategy: {insights['hhi_diff']:+.3f} HHI {'(focused approach works)' if insights['hhi_diff'] > 0.05 else '(balanced approach works)' if insights['hhi_diff'] < -0.05 else '(flexible)'}
  • Top Success Genres: {', '.join(insights['top_genres'])}
  • Top Success Platforms: {', '.join(insights['top_platforms'])}"""
    else:
        doc_content += "\n  [Insufficient data for detailed success analysis]"
    
    doc_content += "\n\n"

doc_content += f"""
STRATEGIC IMPLICATIONS
{'-'*100}

Market Entry Recommendations:
1. Identify target segment based on budget and capabilities
2. Align genre/platform choices with segment characteristics
3. Set quality targets appropriate to segment expectations
4. Adopt regional strategy matching segment distribution
5. Assess competition level before committing resources

Publisher Guidance:
1. Portfolio diversification across multiple segments recommended
2. Segment-specific marketing and positioning strategies essential
3. Quality standards vary significantly by segment
4. Regional focus decisions should align with segment characteristics
5. Publisher reputation impact varies by segment

Investor Considerations:
1. Hit rates vary dramatically across segments ({profile_df['Hit_Rate'].min():.1f}% to {profile_df['Hit_Rate'].max():.1f}%)
2. Average sales range from ${profile_df['Avg_Sales'].min():.2f}M to ${profile_df['Avg_Sales'].max():.2f}M
3. Competition intensity varies by segment
4. Regional concentration affects risk profile
5. Genre and platform choices significantly impact success probability

TECHNICAL METHODOLOGY
{'-'*100}

Algorithm: {algorithm}
Number of Clusters: {n_clusters}
Sample Size: {n_samples:,} games
Silhouette Score: {silhouette_score:.4f} ({'Excellent (>0.55)' if silhouette_score >= 0.55 else 'Good (>0.50)' if silhouette_score >= 0.50 else 'Acceptable (>0.40)' if silhouette_score >= 0.40 else 'Below Target'})

Features Used:
- Sales metrics (total, regional, ratios)
- Quality indicators (critic scores)
- Market structure (HHI, global/regional flags)
- Strategic indicators (hit status, game age)
- Publisher characteristics (reputation, track record)
- Genre and platform attributes

DELIVERABLES
{'-'*100}

1. Clustered dataset: df_with_clusters_enhanced.csv
2. Cluster profiles: cluster_profiles_enhanced.csv
3. Clustering metadata: model_metadata_enhanced.txt
4. Visual analytics: business_analytics_dashboard.png
5. This executive report

CONCLUSION
{'-'*100}

The analysis successfully identified {n_clusters} distinct market segments with clear
characteristics and actionable insights. Each segment exhibits unique patterns in
sales performance, regional distribution, quality requirements, and success factors.

These findings enable data-driven decision-making for game development, publishing,
and investment strategies. Segment-specific recommendations provide concrete guidance
for market entry, positioning, and resource allocation.

The clustering quality (Silhouette: {silhouette_score:.4f}) {'meets' if silhouette_score >= 0.50 else 'approaches'} the
target threshold, indicating {'reliable' if silhouette_score >= 0.50 else 'acceptable'} segment separation and
{'high' if silhouette_score >= 0.55 else 'reasonable'} confidence in the discovered market structure.

{'='*100}
END OF REPORT
{'='*100}
"""

# Save documentation
with open('BUSINESS_DOCUMENTATION_FINAL.txt', 'w', encoding='utf-8') as f:
    f.write(doc_content)

print("Executive documentation saved: BUSINESS_DOCUMENTATION_FINAL.txt")
print(f"Document length: {len(doc_content):,} characters")
print("\nDocumentation generation complete.")

---

## 8. Summary and Conclusions

In [ ]:
# Print final summary
print("\n" + "="*100)
print("ANALYSIS COMPLETE - FINAL SUMMARY")
print("="*100)
print()
print(f"Clustering Results:")
print(f"  Algorithm: {algorithm}")
print(f"  Clusters discovered: {n_clusters}")
print(f"  Games analyzed: {n_samples:,}")
print(f"  Silhouette score: {silhouette_score:.4f} - {'TARGET ACHIEVED' if silhouette_score >= 0.50 else 'ACCEPTABLE PERFORMANCE'}")
print()
print(f"Market Segments:")
for c in sorted(cluster_names.keys()):
    row = profile_df[profile_df['Cluster'] == c].iloc[0]
    print(f"  {c+1}. {cluster_names[c]}")
    print(f"     {int(row['Size']):,} games ({row['Size_%']:.1f}%) | Avg Sales: ${row['Avg_Sales']:.2f}M")
print()
print(f"Deliverables Created:")
print(f"  ✓ Cluster characterization and naming")
print(f"  ✓ Success factor analysis")
print(f"  ✓ Competitive landscape assessment")
print(f"  ✓ Strategic recommendations by segment")
print(f"  ✓ Business analytics dashboard")
print(f"  ✓ Executive documentation report")
print()
print(f"Key Insights:")
print(f"  • Market shows clear segmentation along regional and performance dimensions")
print(f"  • Success factors vary significantly by segment")
print(f"  • Quality impact ranges from critical to limited depending on segment")
print(f"  • Publisher reputation more important in some segments than others")
print(f"  • Regional strategy alignment crucial for success")
print()
print("="*100)
print("Ready for business presentation and strategic decision-making")
print("="*100)